# ZERO KNOWLEDGE PROOFS - STARK OPTIMIZATION

In [ ]:
import math
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
P = 2**64 - 2**32 + 1   # Goldilocks prime


def inverse(a):
    return pow(a, P - 2, P)


def make_domain(size, offset):
    w = pow(7, (P - 1) // size, P)
    return [offset * pow(w, i, P) % P for i in range(size)]


def evaluate_poly(coeffs, x):
    result = 0
    for c in reversed(coeffs):
        result = (result * x + c) % P
    return result


def fold_by_2(values, domain, beta):
    half = len(values) // 2
    new_values = []
    new_domain = []
    for i in range(half):
        x = domain[i]
        a = values[i]          # f(x)
        b = values[i + half]   # f(-x)
        even = (a + b) * inverse(2) % P
        odd = (a - b) * inverse(2 * x) % P
        new_values.append((even + beta * odd) % P)
        new_domain.append(x * x % P)
    return new_values, new_domain


def fold_by_4(values, domain, beta):
    # folding by 4 is two folds by 2, the second one using beta^2
    values, domain = fold_by_2(values, domain, beta)
    return fold_by_2(values, domain, beta * beta % P)


def fold_with_schedule(coeffs, schedule, size=512, seed=0):
    rnd = random.Random(seed)
    # set of input x values
    domain = make_domain(size, offset=7)
    # based on the x values, the corresponding values in the polynomial (based on x, building a set of values for f(x).)
    values = [evaluate_poly(coeffs, x) for x in domain]
    for fold in schedule:
        beta = rnd.randrange(2, P)   # the verifier's random number
        if fold == 2:
            values, domain = fold_by_2(values, domain, beta)
        else:
            values, domain = fold_by_4(values, domain, beta)
    return values


rnd = random.Random(1)
poly = [rnd.randrange(P) for _ in range(256)]

In [ ]:
HASH_BYTES = 32    # size of one hash
BLOCK_BYTES = 64   # bytes one hash call can take in
TRACE_BYTES = 8    # trace values are 64-bit numbers


def layer_info(layer, setup):
    size = setup["domain"] // 2**layer
    if layer == 0:
        return size, setup["width"], TRACE_BYTES
    return size, 1, setup["value_bytes"]


def round_time(layer, fold, setup):
    size, columns, value_bytes = layer_info(layer, setup)
    leaves = size // fold
    hashes_per_leaf = max(1, math.ceil(fold * columns * value_bytes / BLOCK_BYTES))
    hash_work = leaves * hashes_per_leaf + (leaves - 1)

    field_work = 4 * size * (1 - 1 / fold)
    if layer == 0:
        field_work += size * columns   # merging the trace columns
    return hash_work + field_work / setup["hash_cost"]


def merkle_hashes_needed(num_leaves, opened):
    # hashes needed to prove every opened leaf is in the tree (shared paths are sent once)
    count = 0
    nodes = set(opened)
    while num_leaves > 1:
        siblings = {i ^ 1 for i in nodes}
        count += len(siblings - nodes)
        nodes = {i // 2 for i in nodes}
        num_leaves //= 2
    return count


def round_size(layer, fold, setup, queries):
    size, columns, value_bytes = layer_info(layer, setup)
    leaves = size // fold
    opened = {q % leaves for q in queries}
    return (HASH_BYTES
            + value_bytes * columns * fold * len(opened)
            + HASH_BYTES * merkle_hashes_needed(leaves, opened))

In [ ]:
def halvings_done(fold):
    return 1 if fold == 2 else 2


def all_choices(halvings):
    choices = []
    for layer in range(halvings):
        choices.append((layer, 2))
        if layer + 2 <= halvings:
            choices.append((layer, 4))
    return choices


def total_cost(schedule, cost):
    total = 0.0
    layer = 0
    for fold in schedule:
        total += cost[(layer, fold)]
        layer += halvings_done(fold)
    return total


def always(fold, halvings):
    # always fold by 2, or always fold by 4 (with a final 2 if one halving is left over)
    schedule = []
    layer = 0
    while layer < halvings:
        f = 4 if (fold == 4 and layer + 2 <= halvings) else 2
        schedule.append(f)
        layer += halvings_done(f)
    return tuple(schedule)


def best_schedule(halvings, cost):
    # Going from the end of the folding process to the beginning, comparing the fold choices & determining the optimal folding path
    best_cost = [0.0] * (halvings + 1)
    best_plan = [()] * (halvings + 1)
    for layer in range(halvings - 1, -1, -1):
        c = cost[(layer, 2)] + best_cost[layer + 1]
        plan = (2,) + best_plan[layer + 1]
        if layer + 2 <= halvings:
            c4 = cost[(layer, 4)] + best_cost[layer + 2]
            if c4 < c:
                c = c4
                plan = (4,) + best_plan[layer + 2]
        best_cost[layer] = c
        best_plan[layer] = plan
    return best_plan[0]


def blend(setup, size_cost):
    # combined cost = lam * time + (1 - lam) * size, both scaled by "always fold by 2"
    time_cost = {choice: round_time(*choice, setup) for choice in size_cost}
    base = always(2, setup["halvings"])
    time_base = total_cost(base, time_cost)
    size_base = total_cost(base, size_cost)
    lam = setup["lam"]
    return {choice: lam * time_cost[choice] / time_base + (1 - lam) * size_cost[choice] / size_base
            for choice in size_cost}

In [ ]:
def expected_size(layer, fold, setup):
    size, columns, value_bytes = layer_info(layer, setup)
    leaves = size // fold
    t = setup["queries"]
    opened = leaves * (1 - (1 - 1 / leaves) ** t)   # expected number of distinct leaves opened
    hashes = 0.0
    n = leaves
    while n > 1:   # expected Merkle hashes, one tree level at a time
        hashes += n * ((1 - 1 / n) ** t - (1 - 2 / n) ** t)
        n //= 2
    return HASH_BYTES + value_bytes * columns * fold * opened + HASH_BYTES * hashes


def expected_cost(setup):
    size_cost = {choice: expected_size(*choice, setup) for choice in all_choices(setup["halvings"])}
    return blend(setup, size_cost)

In [ ]:
def random_setup(rnd):
    halvings = rnd.randint(4, 16)
    blowup = rnd.choice([2, 4, 8, 16])
    return dict(halvings=halvings,
                blowup=blowup,
                queries=math.ceil(100 / math.log2(blowup)),
                width=int(round(2 ** rnd.uniform(0, 8))),
                value_bytes=rnd.choice([8, 16, 24]),
                hash_cost=2 ** rnd.uniform(math.log2(10), math.log2(300)),
                lam=rnd.random(),
                domain=2**halvings * blowup)


def predicted_label(setup, rnd):
    queries = [rnd.randrange(setup["domain"]) for _ in range(setup["queries"])]
    size_cost = {choice: round_size(*choice, setup, queries) for choice in all_choices(setup["halvings"])}
    return best_schedule(setup["halvings"], blend(setup, size_cost))


def average_label(setup):
    return best_schedule(setup["halvings"], expected_cost(setup))


FEATURE_NAMES = ["layer", "halvings_left", "log2_layer_size", "is_first_layer",
                 "width", "value_bytes", "queries", "log2_blowup", "log2_hash_cost", "lam"]


def features(setup, layer):
    return [layer,
            setup["halvings"] - layer,
            math.log2(setup["domain"] // 2**layer),
            int(layer == 0),
            setup["width"],
            setup["value_bytes"],
            setup["queries"],
            math.log2(setup["blowup"]),
            math.log2(setup["hash_cost"]),
            setup["lam"]]


def make_rows(ids, labels):
    X, y = [], []
    for i in ids:
        layer = 0
        for fold in labels[i]:
            if setups[i]["halvings"] - layer >= 2:   # with 1 halving left, fold by 2 is the only choice
                X.append(features(setups[i], layer))
                y.append(fold)
            layer += halvings_done(fold)
    return np.array(X), np.array(y)


setup_rnd = random.Random(0)
query_rnd = random.Random(1)
setups = [random_setup(setup_rnd) for _ in range(6000)]
predicted_labels = [predicted_label(s, query_rnd) for s in setups]
average_labels = [average_label(s) for s in setups]

mixed = sum(1 for s, label in zip(setups, average_labels) if label != always(4, s["halvings"]))
differ = sum(1 for a, b in zip(predicted_labels, average_labels) if a != b)
print(f"{len(setups)} setups labelled")
print(f"best schedule is a real mix of 2s and 4s in {100 * mixed / len(setups):.1f}% of setups")
print(f"predicted label differs from average label in {100 * differ / len(setups):.1f}% of setups (unlucky draws)")

In [ ]:
train_ids, test_ids = train_test_split(range(len(setups)), test_size=0.25, random_state=0)
train_ids = sorted(train_ids)
test_ids = sorted(test_ids)


def train_model(ids, labels):
    X, y = make_rows(ids, labels)
    model = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.1, random_state=0)
    model.fit(X, y)
    return model


predicted_model = train_model(train_ids, predicted_labels)
average_model = train_model(train_ids, average_labels)

X_test, y_test = make_rows(test_ids, average_labels)

In [ ]:
def predict_schedule(setup, model):
    schedule = []
    layer = 0
    while layer < setup["halvings"]:
        if setup["halvings"] - layer == 1:
            fold = 2
        else:
            fold = int(model.predict(np.array([features(setup, layer)]))[0])
        schedule.append(fold)
        layer += halvings_done(fold)
    return tuple(schedule)


def change(new, old):
    # percent change from old to new (negative = new is cheaper)
    return 100 * (new / old - 1)


def compare(model, ids):
    results = []
    for i in ids:
        s = setups[i]
        cost = expected_cost(s)
        time = {choice: round_time(*choice, s) for choice in cost}
        size = {choice: expected_size(*choice, s) for choice in cost}
        ml = predict_schedule(s, model)
        best = average_labels[i]
        all2 = always(2, s["halvings"])
        all4 = always(4, s["halvings"])
        results.append(dict(
            exact=(ml == best),
            mixed=(best != all4),
            regret=change(total_cost(ml, cost), total_cost(best, cost)),
            cost_vs_2=change(total_cost(ml, cost), total_cost(all2, cost)),
            cost_vs_4=change(total_cost(ml, cost), total_cost(all4, cost)),
            best_vs_2=change(total_cost(best, cost), total_cost(all2, cost)),
            best_vs_4=change(total_cost(best, cost), total_cost(all4, cost)),
            time_vs_2=change(total_cost(ml, time), total_cost(all2, time)),
            time_vs_4=change(total_cost(ml, time), total_cost(all4, time)),
            size_vs_2=change(total_cost(ml, size), total_cost(all2, size)),
            size_vs_4=change(total_cost(ml, size), total_cost(all4, size)),
            schedule=ml, all4=all4))
    return results


def mean(results, key, only_mixed=False):
    return np.mean([r[key] for r in results if r["mixed"] or not only_mixed])


predicted_results = compare(predicted_model, test_ids)
average_results = compare(average_model, test_ids)

In [ ]:
print("examples of the model's mixed schedules:")
mixed_results = [r for r in average_results if r["schedule"] != r["all4"]]
for r in sorted(mixed_results, key=lambda r: r["cost_vs_4"])[:5]:
    print(f"   model {str(r['schedule']):26s} vs always 4 {str(r['all4']):22s} -> {r['cost_vs_4']:+.2f}% cost")

In [ ]:
halving_counts = sorted(set(setups[i]["halvings"] for i in test_ids))
plot = {name: {"cost": [], "rounds": []} for name in ["always fold by 2", "always fold by 4", "mixed (model)"]}

for h in halving_counts:
    ids = [i for i in test_ids if setups[i]["halvings"] == h]
    for name in plot:
        costs, rounds = [], []
        for i in ids:
            s = setups[i]
            if name == "always fold by 2":
                schedule = always(2, h)
            elif name == "always fold by 4":
                schedule = always(4, h)
            else:
                schedule = predict_schedule(s, average_model)
            costs.append(total_cost(schedule, expected_cost(s)))
            rounds.append(len(schedule))
        plot[name]["cost"].append(np.mean(costs))
        plot[name]["rounds"].append(np.mean(rounds))

styles = {"always fold by 2": ("tab:blue", "o", "-"),
          "always fold by 4": ("indianred", "s", "-"),
          "mixed (model)": ("seagreen", "D", "--")}

fig, graph = plt.subplots(1, 1, figsize=(13, 4.5))
for name, (color, marker, line) in styles.items():
    graph.plot(halving_counts, plot[name]["cost"], color=color, marker=marker, linestyle=line, label=name)

graph.set_xlabel("round")
graph.set_ylabel("cost (always fold by 2 = 1.0)")
graph.set_title("Cost: mixed vs fixed folding", fontweight="bold")
graph.legend()
graph.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"{'schedule':18s} | {'average cost':>12}")
print("-" * 52)
for name in plot:
    all_costs, all_rounds = [], []
    for i in test_ids:
        s = setups[i]
        if name == "always fold by 2":
            schedule = always(2, s["halvings"])
        elif name == "always fold by 4":
            schedule = always(4, s["halvings"])
        else:
            schedule = predict_schedule(s, average_model)
        all_costs.append(total_cost(schedule, expected_cost(s)))
    print(f"{name:18s} | {np.mean(all_costs):>12.3f}")